In [2]:
!pip install -q langchain-classic langchain_openai langchain-core langchain-community langgraph langchain-tavily youtube-search

In [3]:
import os

with open("/content/api_key_openia.txt", "r", encoding="utf-8-sig") as archivo:
  apikey = archivo.readline().strip()
  os.environ["OPENAI_API_KEY"] = apikey

with open("/content/api_tavily.txt", "r", encoding="utf-8-sig") as archivo:
  apikey = archivo.readline().strip()
  os.environ["TAVILY_API_KEY"] = apikey

# **Herramienta 1: Idenficador de gaps de conocimiento**

In [5]:
from typing import Optional
from langchain_core.tools import BaseTool
from pydantic import BaseModel, Field
from langchain_core.callbacks import (AsyncCallbackManagerForToolRun, CallbackManagerForToolRun)

# Definición de los tipos de campo:
class SkillGapInput(BaseModel):
    perfil_actual: str = Field(description="Habilidades y formación actual del usuario")
    objetivo_hacer: str = Field(description="La capacidad o proyecto que el usuario desea lograr")

class SkillGapTool(BaseTool):
    name: str = "skill_gap_analyzer"
    description: str = "Analiza qué habilidades técnicas faltan para lograr un objetivo específico."
    args_schema: type[BaseModel] = SkillGapInput

    def _run(self, perfil_actual: str, objetivo_hacer: str, run_manager: Optional[CallbackManagerForToolRun] = None) -> str:
        """Ejecución síncrona"""
        return f"Analiza como experto qué conceptos técnicos le faltan a un {perfil_actual} para hacer {objetivo_hacer}. Sé específico con librerías y conceptos de IA."

    async def _arun(self, perfil_actual: str, objetivo_hacer: str, run_manager: Optional[AsyncCallbackManagerForToolRun] = None) -> str:
        """Ejecución asíncrona"""
        return self._run(perfil_actual, objetivo_hacer, run_manager=run_manager.get_sync())

# **Herramienta 2: Idenficador de Recursos y Expertos**


Uso de herramienta externa: Tavily

In [6]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

# Definición de clases
class SearchInput(BaseModel):
    tema: str = Field(description="El tema o tecnología a investigar en la web")

# Clase de la Herramienta
class DeepExpertHunter(BaseTool):
    name: str = "deep_expert_hunter"
    description: str = "Busca en la web y resume los 3 mejores recursos o tutoriales sobre un tema."
    args_schema: type[BaseModel] = SearchInput

    def _run(self, tema: str, run_manager: Optional[CallbackManagerForToolRun] = None) -> str:
        search = TavilySearchResults(max_results=5)
        resultados = search.run(tema)

        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        parser = StrOutputParser()

        prompt = ChatPromptTemplate.from_messages([
            ("system", "Eres un analista de formación tech. Resume los 3 mejores enlaces de estos resultados para una persona no experta aprendiendo sobre tecnología."),
            ("human", "Resultados para '{tema}':\n\n{resultados}")
        ])

        # Elaboración de cadena
        chain = prompt | llm | parser
        return chain.invoke({"tema": tema, "resultados": resultados})

    async def _arun(self, tema: str, run_manager: Optional[AsyncCallbackManagerForToolRun] = None) -> str:
        return self._run(tema, run_manager=run_manager.get_sync())

# **Herramienta 3: Idenficador de tutoriales**


Uso de herramienta externa: YouTube Search Tool

In [7]:
from langchain_community.tools import YouTubeSearchTool

class YouTubeInput(BaseModel):
    query: str = Field(description="El tema o tutorial específico a buscar en YouTube")

class YouTubeHunterTool(BaseTool):
    name: str = "youtube_hunter"
    description: str = "Busca videos y tutoriales en YouTube sobre temas tecnológicos específicos."
    args_schema: type[BaseModel] = YouTubeInput

    def _run(self, query: str, run_manager: Optional[CallbackManagerForToolRun] = None) -> str:
        """Búsqueda en YouTube."""
        # Herramienta externa: API Youtube
        yt_search = YouTubeSearchTool()

        # Número de resultados
        resultados = yt_search.run(f"{query},3")

        return f"Aquí tienes los videos más relevantes encontrados en YouTube para '{query}':\n{resultados}"

    async def _arun(self, query: str, run_manager: Optional[AsyncCallbackManagerForToolRun] = None) -> str:
        """Ejecución asíncrona."""
        return self._run(query, run_manager=run_manager.get_sync())

# **Herramienta 4: Ideas de proyectos**


In [8]:
class ProjectInput(BaseModel):
    idea_proyecto: str = Field(description="Una idea creativa de proyecto que una el perfil del usuario con la tecnología")
    tecnologia_clave: str = Field(description="La tecnología que el usuario acaba de aprender")
    perfil_usuario: str = Field(description="El perfil del usuario")

class ProjectArchitectTool(BaseTool):
    name: str = "generador_proyectos"
    description: str = "Diseña una idea de proyecto práctico. Debes proponer una idea creativa basada en el perfil."
    args_schema: type[BaseModel] = ProjectInput

    def _run(self, idea_proyecto: str, tecnologia_clave: str, perfil_usuario: str, run_manager: Optional[CallbackManagerForToolRun] = None) -> str:
        """Lógica para proponer un proyecto dinámico."""
        return f"""
        Propuesta de Proyecto para un {perfil_usuario}:
        ---
        Idea: {idea_proyecto}
        Descripción: Un sistema basado en {tecnologia_clave} diseñado para potenciar el perfil de {perfil_usuario}.
        Valor: Este proyecto demuestra que puedes se puede aplicar lo aprendido en tu campo profesional u objetivo deseado.
        """

**Set de 4 herramientas**

In [9]:
gap_analyzer = SkillGapTool()
resource_hunter = DeepExpertHunter()
tutorial_finder = YouTubeHunterTool()
project_architect = ProjectArchitectTool()

toolkit = [gap_analyzer, resource_hunter, tutorial_finder, project_architect]

**Creación de agente**

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent

llm = ChatOpenAI(temperature=0)
memory = MemorySaver()

system_prompt = """Eres el 'Mentor Inteligente de Conocimientos de IA'. Tu objetivo es ayudar a los usuarios
                  a profundizar su conocimiento de IA mediante un flujo de trabajo de 4 pasos:

                  1. Identificar qué habilidades les faltan (Usa: skill_gap_analyzer).
                  2. Buscar fuentes y documentación en la web (Usa: deep_expert_hunter).
                  3. Encontrar tutoriales prácticos en video (Usa: identificador_tutoriales).
                  4. Realiza una propuesta de un proyecto innovador de acuerdo al perfil (Usa: generador_proyectos).

                  Sé directo, profesional y proporciona siempre los enlaces encontrados por las herramientas."""

agent =create_agent(model=llm,tools=toolkit,system_prompt=system_prompt,checkpointer=memory)


**Prueba de agente**

In [ ]:
from langchain_core.messages import HumanMessage

# Configuración del hilo para la memoria
config = {"configurable": {"thread_id": "prueba1"}}

# Prompt de prueba #1
input_message = {"messages": [HumanMessage(
    content=
    "Hola, soy economista y quiero aprender a crear agentes de IA para análisis financiero. "
     "Analiza mi brecha, busca recursos de lectura y dame una idea de proyecto.")]}

# Ejecución en streaming para ver el razonamiento paso a paso
for step in agent.stream(input_message, config, stream_mode="values"):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Hola, soy economista y quiero aprender a crear agentes de IA para análisis financiero. Analiza mi brecha, busca recursos de lectura y dame una idea de proyecto.
================================== Ai Message ==================================
Tool Calls:
  skill_gap_analyzer (call_0vgZEuFBDNRmVSGXeawxuLvk)
 Call ID: call_0vgZEuFBDNRmVSGXeawxuLvk
  Args:
    perfil_actual: economista
    objetivo_hacer: crear agentes de IA para análisis financiero
================================= Tool Message =================================
Name: skill_gap_analyzer

Analiza como experto qué conceptos técnicos le faltan a un economista para hacer crear agentes de IA para análisis financiero. Sé específico con librerías y conceptos de IA.
================================== Ai Message ==================================

Para crear agentes de IA para análisis financiero, como economista, te recomendaría aprender sobre los si

In [ ]:
# Prompt de prueba #2: Memoria
config = {"configurable": {"thread_id": "prueba1"}}
input_message = {"messages": [HumanMessage(
    content=
    "Teniendo en cuenta mi profesión y el conocimiento que quiero adquirir, sugiéreme algunos tutoriales de Youtube")]}

for step in agent.stream(input_message, config, stream_mode="values"):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Teniendo en cuenta mi profesión y el conocimiento que quiero adquirir, sugiéreme algunos tutoriales de Youtube
================================== Ai Message ==================================
Tool Calls:
  youtube_hunter (call_GPbJfqFRm4Nsn3xNSmNAONRs)
 Call ID: call_GPbJfqFRm4Nsn3xNSmNAONRs
  Args:
    query: Machine Learning for Financial Analysis tutorial
  youtube_hunter (call_ICcdI4sfKojvgXwqvVM8ezUz)
 Call ID: call_ICcdI4sfKojvgXwqvVM8ezUz
  Args:
    query: TensorFlow tutorial for beginners
================================= Tool Message =================================
Name: youtube_hunter

Aquí tienes los videos más relevantes encontrados en YouTube para 'TensorFlow tutorial for beginners':
['https://www.youtube.com/watch?v=i8NETqtGHms&pp=ygUhVGVuc29yRmxvdyB0dXRvcmlhbCBmb3IgYmVnaW5uZXJz', 'https://www.youtube.com/watch?v=tPYj3fFJGjk&pp=ygUhVGVuc29yRmxvdyB0dXRvcmlhbCBmb3IgYmVnaW5uZXJz', 'https://w